# L13d: Inspect a Local Urea-Cycle MCP Server

This lab starts with the pure flux-balance computation, then crosses the protocol boundary through the official Python SDK's in-memory client.

> **Learning objectives**
>
> - Test the numerical core without MCP.
> - Discover a read-only resource and typed tools.
> - Exercise valid, malformed, and unsupported calls.
> - Explain why protocol success is not numerical validation.


## Setup


In [1]:
from pathlib import Path
import json
import sys

week_root = Path("..").resolve()
source_dir = week_root / "python" / "src"
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

from cheme5800_mcp.network import DATA_PATH, check_flux_balance, summarize_network
payload = json.loads(DATA_PATH.read_text(encoding="utf-8"))
summarize_network()


{'network_id': 'urea-cycle',
 'name': 'HL-60 urea-cycle teaching network',
 'number_of_species': 18,
 'number_of_reactions': 19,
 'species': ['M_ATP_c',
  'M_L-Citrulline_c',
  'M_L-Aspartate_c',
  'M_AMP_c',
  'M_Diphosphate_c',
  'M_N-(L-Arginino)succinate_c',
  'M_Fumarate_c',
  'M_L-Arginine_c',
  'M_H2O_c',
  'M_L-Ornithine_c',
  'M_Urea_c',
  'M_Carbamoyl_phosphate_c',
  'M_Orthophosphate_c',
  'M_Oxygen_c',
  'M_NADPH_c',
  'M_H_c',
  'M_Nitric_oxide_c',
  'M_NADP_c'],
 'reactions': ['v1',
  'v2',
  'v3',
  'v4',
  'v5',
  'b1',
  'b2',
  'b3',
  'b4',
  'b5',
  'b6',
  'b7',
  'b8',
  'b9',
  'b10',
  'b11',
  'b12',
  'b13',
  'b14']}

## Part 1 — Validate the pure computation


In [2]:
balanced = check_flux_balance(payload["reference_fluxes"]["balanced"])
imbalanced = check_flux_balance(payload["reference_fluxes"]["imbalanced"])
{
    "balanced": balanced["is_balanced"],
    "balanced_max_residual": balanced["max_abs_residual"],
    "imbalanced": imbalanced["is_balanced"],
    "imbalanced_max_residual": imbalanced["max_abs_residual"],
}


{'balanced': True,
 'balanced_max_residual': 0.0,
 'imbalanced': False,
 'imbalanced_max_residual': 1.0}

The imbalanced fixture changes the urea exchange flux. The residual is reported in species order, so the failure remains interpretable rather than collapsing to a Boolean.


In [3]:
residuals = dict(zip(imbalanced["species_order"], imbalanced["residual"], strict=True))
{species: value for species, value in residuals.items() if value != 0.0}


{'M_Urea_c': 1.0}

## Part 2 — Discover and call the MCP capabilities


In [4]:
from mcp import Client
from cheme5800_mcp.server import mcp

async with Client(mcp) as client:
    protocol = client.protocol_version
    tools = await client.list_tools()
    resources = await client.list_resources()
    summary_result = await client.call_tool("summarize_network", {})
    balance_result = await client.call_tool(
        "check_flux_balance",
        {"flux": payload["reference_fluxes"]["balanced"]},
    )
    malformed_result = await client.call_tool("check_flux_balance", {"flux": [0.0, 1.0]})
    unsupported_result = await client.call_tool("delete_network", {})

{
    "protocol": protocol,
    "tools": sorted(tool.name for tool in tools.tools),
    "resources": [str(resource.uri) for resource in resources.resources],
    "summary_error": summary_result.is_error,
    "balanced": balance_result.structured_content["is_balanced"],
    "malformed_rejected": malformed_result.is_error,
    "unsupported_rejected": unsupported_result.is_error,
}


{'protocol': '2026-07-28',
 'tools': ['check_flux_balance', 'summarize_network'],
 'resources': ['cheme://metabolic-network/urea-cycle'],
 'summary_error': False,
 'balanced': True,
 'malformed_rejected': True,
 'unsupported_rejected': True}

## Part 3 — Interpret the boundary

- The pure tests establish the numerical meaning of `S*v`.
- The client test establishes discovery, schemas, serialization, and error propagation.
- The missing `delete_network` tool demonstrates fail-closed capability design.
- Local standard I/O avoids a network listener and requires no model account.

MCP Inspector may be used for the same discovery/call sequence. Its exact command is revalidated immediately before release because SDK tooling changes faster than the learning objectives.
